# 06 · 獎勵塑形與環境包裝器

有些環境的獎勵**很稀疏**——比如 MountainCar,小車要爬上山才給獎勵,但隨機亂動幾乎永遠到不了山頂,agent 收不到任何訊號、根本學不動。

兩個救星:
- **環境包裝器 wrapper**:像洋蔥一樣一層層套在環境外,改觀察、改獎勵、限制步數,而**不動原環境**。
- **獎勵塑形 reward shaping**:自己加一點「中途獎勵」引導 agent,把稀疏訊號變稠密。

## 1. 安裝

In [ ]:
%pip install -q -U "gymnasium[classic-control]"

## 2. 內建包裝器:統計與步數限制

`RecordEpisodeStatistics` 自動記錄每回合長度與報酬;`TimeLimit` 設定步數上限。包裝器可以一層層疊。

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics, TimeLimit

env = gym.make("MountainCar-v0")
env = TimeLimit(env, max_episode_steps=200)
env = RecordEpisodeStatistics(env)

obs, info = env.reset(seed=0)
print("觀察:[位置, 速度] =", obs)
print("原始獎勵設計:每一步 -1,到山頂才結束 → 非常稀疏")

## 3. 自訂 RewardWrapper:加中途獎勵

繼承 `gym.RewardWrapper`,改寫 `reward()`。這裡用「能量」(位置高度 + 速度平方)當塑形訊號:車子爬越高、衝越快,就多給一點獎勵,引導它學會來回擺盪蓄能。

In [ ]:
class EnergyShaping(gym.Wrapper):
    """把稀疏的 MountainCar 獎勵,加上一點基於『能量』的中途獎勵。"""

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        position, velocity = obs
        # 位置越高(越接近山頂 0.5)+ 速度越大 → 額外獎勵
        shaping = (position + 0.5) + 10.0 * (velocity ** 2)
        return obs, reward + shaping, terminated, truncated, info


base = TimeLimit(gym.make("MountainCar-v0"), max_episode_steps=200)
shaped = EnergyShaping(base)

# 比較:同樣隨機亂走,塑形後每回合拿到的訊號稠密多了
def run(env):
    obs, _ = env.reset(seed=0); total, done = 0.0, False
    while not done:
        obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
        total += r; done = terminated or truncated
    return total

print("原始獎勵總和  :", round(run(base), 1))
print("塑形後獎勵總和:", round(run(shaped), 1), "← 中途就有訊號可學")

## 4. ObservationWrapper:改觀察

同理,`ObservationWrapper` 可以正規化觀察值——很多演算法吃正規化後的輸入更穩。

In [ ]:
class NormalizeObs(gym.ObservationWrapper):
    """把 MountainCar 的觀察(位置、速度)線性縮放到大約 0~1。"""

    def observation(self, obs):
        low, high = self.observation_space.low, self.observation_space.high
        return (obs - low) / (high - low)


norm_env = NormalizeObs(gym.make("MountainCar-v0"))
obs, _ = norm_env.reset(seed=0)
print("正規化後的觀察:", np.round(obs, 3))

## 小結

- **Wrapper 像洋蔥**,一層層套在環境外改行為,不動原環境:`TimeLimit`、`RewardWrapper`、`ObservationWrapper`…
- **獎勵塑形**把稀疏訊號變稠密,讓 agent 學得動——但要小心別塑形成「鑽漏洞」的壞策略。
- 這些是把 RL 用在**真實問題**上的關鍵手藝。

下一課:把整套合起來——**親手把一個小遊戲包成 gymnasium 環境**。